# ==============================================================================
# 🛡️ CYBERSECURITY THREAT INTELLIGENCE: PHISHING DETECTION VIA BERNOULLI NAIVE BAYES
# ==============================================================================
### Formulation:
In cybersecurity URL heuristics, features represent discrete boolean security checks:
e.g., `having_IP_Address` $\in \{0, 1\}$, `having_At_Symbol` $\in \{0, 1\}$, `SSLfinal_State` $\in \{0, 1\}$.
Bernoulli Naive Bayes calculates log odds ratio:
$$\log \frac{P(\text{Phishing} \mid X)}{P(\text{Legitimate} \mid X)} = \log \frac{P(\text{Phishing})}{P(\text{Legitimate})} + \sum_{i=1}^d \left[ x_i \log \frac{p_{1i}}{p_{0i}} + (1 - x_i) \log \frac{1 - p_{1i}}{1 - p_{0i}} \right]$$


In [1]:
# STEP 1: IMPORTS & ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import Binarizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Cybersecurity ML Environment initialized.")


✅ STEP 1: Cybersecurity ML Environment initialized.


## 📥 STEP 2 & 3: INGESTION & COLUMN SANITIZATION
Loading Kaggle Phishing Websites dataset with 30 security heuristic indicators.


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/phishing/Phishing_Websites_Data.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().replace(' ', '_') for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} URLs, {df.shape[1]} attributes")
print(f"Target distribution:\n{df['Result'].value_counts().to_dict()}")
df.head(3)


📊 Dataset Shape: 11,055 URLs, 31 attributes
Target distribution:
{1: 6157, -1: 4898}


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1


## 🧹 STEP 4 & 5: BINARY SEGREGATION & HYGIENE
Converting features from {-1, 0, 1} to binary indicators $\{0, 1\}$ and mapping target Result (-1 -> 0, 1 -> 1).


In [3]:
# STEP 4 & 5: SEGREGATION & BINARIZATION
# Map target -1 (phishing) to 1, 1 (legitimate) to 0 or vice versa
y = df['Result'].map({-1: 1, 1: 0})  # 1 = Phishing, 0 = Legitimate

# Binarize features: values > 0 become 1, <= 0 become 0
X_raw = df.drop(columns=['Result'])
X = (X_raw > 0).astype(int)

print(f"Missing attributes: {X.isnull().sum().sum()}")
print(f"Target Balance (1=Phishing):\n{y.value_counts(normalize=True).round(4) * 100}")


Missing attributes: 0
Target Balance (1=Phishing):
Result
0    55.69
1    44.31
Name: proportion, dtype: float64


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train URLs : {X_train.shape[0]:,}")
print(f"Test URLs  : {X_test.shape[0]:,}")


Train URLs : 8,844
Test URLs  : 2,211


## ⚙️ STEP 7: BERNOULLI PREPROCESSING PIPELINE
Since features are strictly $x_i \in \{0, 1\}$, we pass through directly to `BernoulliNB(binarize=None)`.


In [5]:
# STEP 7: DIRECT BERNOULLI PIPELINE
pipe_bnb = Pipeline([
    ('bnb', BernoulliNB(binarize=None, alpha=1.0))
])
print("✅ STEP 7: Binary Feature Pipeline ready.")


✅ STEP 7: Binary Feature Pipeline ready.


## ⚠️ STEP 8: BASELINE PIPELINE
Training baseline BernoulliNB on 30 boolean URL security checks.


In [6]:
# STEP 8: BASELINE EVALUATION
pipe_bnb.fit(X_train, y_train)
y_pred_base = pipe_bnb.predict(X_test)

print(f"Baseline BernoulliNB Accuracy : {accuracy_score(y_test, y_pred_base)*100:.2f}%")
print(f"Baseline BernoulliNB ROC-AUC  : {roc_auc_score(y_test, pipe_bnb.predict_proba(X_test)[:, 1])*100:.2f}%")


Baseline BernoulliNB Accuracy : 91.36%
Baseline BernoulliNB ROC-AUC  : 96.64%


## 🎯 STEP 9: TUNING SMOOTHING PARAMETER $\alpha$
Optimizing $\alpha$ via Stratified 5-Fold Cross Validation.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'bnb__alpha': np.logspace(-3, 1, 15)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    pipe_bnb,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Alpha Parameter: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Alpha Parameter: {'bnb__alpha': np.float64(0.001)}
🎯 Best 5-Fold CV ROC-AUC: 96.47%


## ⚔️ STEP 10: ALGORITHM DUEL (BernoulliNB vs Logistic Regression)
Evaluating BernoulliNB against Logistic Regression on binary cybersecurity features.


In [8]:
# STEP 10: MODEL BENCHMARK
pipe_lr = Pipeline([
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

y_pred_bnb = best_model.predict(X_test)
y_pred_lr = pipe_lr.predict(X_test)

print("="*65)
print(f"BernoulliNB (Tuned) -> Test Accuracy: {accuracy_score(y_test, y_pred_bnb)*100:.2f}%, ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])*100:.2f}%")
print(f"Logistic Regression -> Test Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%, ROC-AUC: {roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:, 1])*100:.2f}%")
print("="*65)


BernoulliNB (Tuned) -> Test Accuracy: 91.36%, ROC-AUC: 96.65%
Logistic Regression -> Test Accuracy: 93.08%, ROC-AUC: 97.53%


## 📊 STEP 11: EVALUATION REPORT & CONFUSION MATRIX
Cybersecurity threat detection precision and recall.


In [9]:
# STEP 11: EVALUATION REPORT
print("📋 PHISHING DETECTION CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_bnb, target_names=['Legitimate URL', 'Phishing Threat']))

cm = confusion_matrix(y_test, y_pred_bnb)
print(f"Confusion Matrix:\n{cm}")


📋 PHISHING DETECTION CLASSIFICATION REPORT:
                 precision    recall  f1-score   support

 Legitimate URL       0.92      0.92      0.92      1231
Phishing Threat       0.90      0.91      0.90       980

       accuracy                           0.91      2211
      macro avg       0.91      0.91      0.91      2211
   weighted avg       0.91      0.91      0.91      2211

Confusion Matrix:
[[1133   98]
 [  93  887]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & FIREWALL SMOKE TEST
Serializing model and evaluating network firewall gateway latency (<0.5ms).


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/phishing_bernoulli_nb_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_url_features = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_threat = loaded_pipe.predict(sample_url_features)[0]
threat_prob = loaded_pipe.predict_proba(sample_url_features)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🛡️ LIVE FIREWALL GATEWAY AUDIT:")
print(f"   URL Threat Assessment: {'🚨 PHISHING ATTACK DETECTED (BLOCK)' if pred_threat == 1 else '✅ LEGITIMATE TRAFFIC (ALLOW)'}")
print(f"   Threat Probability   : {threat_prob*100:.2f}%")
print(f"   Gateway Latency      : {latency_ms:.3f} ms (SLA < 1.0ms: {'PASS' if latency_ms < 1.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/phishing_bernoulli_nb_pipeline.joblib (2,632 bytes)

🛡️ LIVE FIREWALL GATEWAY AUDIT:
   URL Threat Assessment: 🚨 PHISHING ATTACK DETECTED (BLOCK)
   Threat Probability   : 99.71%
   Gateway Latency      : 3.433 ms (SLA < 1.0ms: CHECK)
